# AsistIA — Asesor Experto en IA Generativa

**Proyecto Final · Módulo de IA Generativa**

**Stack:** Google Gemini · ChromaDB · LangGraph · LangChain
**Dominio:** Asesor experto en Inteligencia Artificial Generativa (ingeniería de prompts, modelos LLM, RAG, agentes)

---

## Contenido del notebook

Este notebook implementa un agente conversacional RAG completo. La estructura sigue el orden de construcción del sistema:

1. **Configuración** — carga de la API key de Gemini desde `.env`.
2. **Base de conocimiento vectorial** — carga de 3 PDFs, chunking, embeddings con Gemini e indexado en ChromaDB.
3. **System prompt** — definición y justificación del rol del agente.
4. **Agente LangGraph** — construcción del grafo con nodos de recuperación y generación, más memoria por `thread_id`.
5. **Función de chat e interacción** — wrapper de uso e impresión de las conversaciones.
6. **Demo de 5 ejemplos documentados** + **prueba explícita de memoria conversacional**.
7. **Celda interactiva** para conversar libremente con el agente.
8. **Diagnóstico** de la base de conocimiento.

## 0. Instalación de dependencias

Las dependencias están listadas en `requirements.txt`. La línea siguiente está comentada para ejecuciones repetidas; descomentar la primera vez.

In [1]:
# Descomentar solo en la primera ejecución del entorno:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuración de la API Key de Gemini

La clave se lee de un archivo `.env` ubicado en la misma carpeta que el notebook. El archivo `.env` no se incluye en el repositorio (está en `.gitignore`); en su lugar se proporciona `.env.example` como plantilla.



In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# find_dotenv() busca el archivo .env subiendo por el árbol de directorios
# desde el directorio actual. Si no lo encuentra, devuelve cadena vacía.
ruta_env = find_dotenv(usecwd=True)
load_dotenv(ruta_env)

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "No se ha encontrado la variable GEMINI_API_KEY.\n"
        "Comprobaciones:\n"
        "  1. Existe un archivo llamado exactamente '.env' (con punto inicial) "
        "en la carpeta del notebook.\n"
        "  2. El archivo contiene la línea: GEMINI_API_KEY=tu_clave\n"
        "  3. La clave es válida (https://aistudio.google.com/app/apikey)."
    )

print(f"API Key cargada correctamente (****{API_KEY[-4:]})")
print(f"Origen: {ruta_env or '(variable de entorno del sistema)'}")

API Key cargada correctamente (****dg90)
Origen: C:\Users\ineja\Desktop\EVOLVEL_RAG\.env


## 2. Construcción de la Base de Conocimiento Vectorial

La base de conocimiento del agente se compone de tres documentos PDF redactados específicamente para el proyecto (~20 páginas en total):

| Documento | Contenido |
|---|---|
| `01_ingenieria_de_prompts.pdf` | Zero-shot, few-shot, Chain of Thought, ReAct, role prompting, buenas prácticas, seguridad |
| `02_modelos_ia_generativa.pdf` | GPT-4o, Claude 3.5, Gemini 2.5, LLaMA 3, Mistral, criterios de selección, embeddings |
| `03_rag_agentes_arquitecturas.pdf` | Pipeline RAG, ChromaDB, chunking, agentes, LangChain, LangGraph, MemorySaver |

**Pipeline de indexado:**

```
PDFs  →  Chunking (RecursiveCharacterTextSplitter)  →  Embeddings (Gemini)  →  ChromaDB (persistente)
```

La colección persiste en disco en `./chroma_db/`, por lo que solo es necesario reindexar cuando cambian los documentos.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

DOCS_DIR = "docs"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "asesor_ia_generativa"

pdfs = [
    os.path.join(DOCS_DIR, "01_ingenieria_de_prompts.pdf"),
    os.path.join(DOCS_DIR, "02_modelos_ia_generativa.pdf"),
    os.path.join(DOCS_DIR, "03_rag_agentes_arquitecturas.pdf"),
]

# Carga de documentos
print("Cargando documentos...")
todos_los_docs = []
for pdf_path in pdfs:
    docs = PyPDFLoader(pdf_path).load()
    todos_los_docs.extend(docs)
    print(f"  - {os.path.basename(pdf_path)}: {len(docs)} páginas")

print(f"\nTotal páginas cargadas: {len(todos_los_docs)}")

Cargando documentos...


  - 01_ingenieria_de_prompts.pdf: 3 páginas
  - 02_modelos_ia_generativa.pdf: 3 páginas
  - 03_rag_agentes_arquitecturas.pdf: 3 páginas

Total páginas cargadas: 9


### 2.1. Chunking

Se utiliza `RecursiveCharacterTextSplitter` con tamaño de chunk de 800 caracteres y solapamiento de 80. El splitter intenta cortar primero por dobles saltos de línea (párrafos), luego por saltos simples y, en último término, por espacios. Esto preserva la coherencia semántica dentro de cada chunk.

El solapamiento del 10 % evita perder información cuando un concepto cae justo en la frontera entre dos chunks consecutivos.

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(todos_los_docs)

print(f"Chunks generados: {len(chunks)}")
print(f"Tamaño medio: {sum(len(c.page_content) for c in chunks) // len(chunks)} caracteres")
print(f"\nEjemplo (primer chunk):")
print("-" * 60)
print(chunks[0].page_content[:300] + "...")
print(f"[Fuente: {os.path.basename(chunks[0].metadata.get('source', 'N/A'))}, "
      f"pág. {chunks[0].metadata.get('page', 'N/A')}]")

Chunks generados: 28
Tamaño medio: 657 caracteres

Ejemplo (primer chunk):
------------------------------------------------------------
Ingenieria de Prompts: Guia Completa
 Tecnicas, estrategias y buenas practicas para comunicarse con LLMs
1. Que es la Ingenieria de Prompts
La ingenieria de prompts es la disciplina que estudia como formular instrucciones efectivas
para los modelos de lenguaje grande (LLMs). Un prompt bien disenado ...
[Fuente: 01_ingenieria_de_prompts.pdf, pág. 0]


### 2.2. Embeddings con Gemini e indexado en ChromaDB

Cada chunk se convierte en un vector mediante el modelo de embeddings de Google (`models/gemini-embedding-001`) y se almacena en ChromaDB con persistencia en disco. La primera ejecución crea la colección; las siguientes reutilizan los embeddings ya indexados (siempre que el directorio `chroma_db/` exista).

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

print("Indexando en ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print(f"\nBase de conocimiento creada:")
print(f"  - Colección: '{COLLECTION_NAME}'")
print(f"  - Vectores indexados: {vectorstore._collection.count()}")
print(f"  - Persistencia: ./{CHROMA_DIR}/")

Indexando en ChromaDB...



Base de conocimiento creada:
  - Colección: 'asesor_ia_generativa'
  - Vectores indexados: 28
  - Persistencia: ./chroma_db/


### 2.3. Verificación de la recuperación

Antes de conectar el agente, se prueba que el vectorstore responde correctamente a una consulta. Se configura un retriever que devuelve los `k=3` chunks más similares por similitud coseno.

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

consulta_prueba = "¿Qué es la técnica Chain of Thought?"
resultados = retriever.invoke(consulta_prueba)

print(f"Consulta de prueba: '{consulta_prueba}'")
print(f"Chunks recuperados: {len(resultados)}\n")
for i, doc in enumerate(resultados, 1):
    fuente = os.path.basename(doc.metadata.get("source", "N/A"))
    pagina = doc.metadata.get("page", "?")
    print(f"[{i}] {fuente} (pág. {pagina})")
    print(f"    {doc.page_content[:200]}...\n")

Consulta de prueba: '¿Qué es la técnica Chain of Thought?'
Chunks recuperados: 3

[1] 01_ingenieria_de_prompts.pdf (pág. 1)
    Incluir dos o mas ejemplos de entrada-salida antes de la consulta real mejora drasticamente la
precision del modelo en tareas con formato especifico. Los ejemplos actuan como un 'molde'
que el modelo ...

[2] 01_ingenieria_de_prompts.pdf (pág. 1)
    3.4 Role Prompting (System Prompt)
Asignar un rol especifico al modelo modifica su comportamiento, tono y nivel de detalle.
Ejemplos de roles efectivos: 'Eres un experto en ciberseguridad con 20 anhos...

[3] 03_rag_agentes_arquitecturas.pdf (pág. 1)
    abstracciones para modelos (ChatOpenAI, ChatGoogleGenerativeAI), prompts
(PromptTemplate, ChatPromptTemplate), recuperadores (VectorStoreRetriever), cadenas
(LCEL - LangChain Expression Language) y ag...



## 3. System Prompt

El system prompt fija la identidad, el comportamiento y los límites del agente. Las decisiones de diseño tomadas para este prompt están documentadas en el `README.md`. En síntesis:

- **Rol concreto y especializado** — evita respuestas genéricas; el agente sabe en qué dominio es experto.
- **Obligación de basarse en el contexto recuperado** — reduce alucinaciones forzando al modelo a usar los documentos como fuente principal.
- **Comportamiento explícito ante falta de contexto** — el agente reconoce sus límites en lugar de inventar información.
- **Idioma fijado en español** — independencia del idioma de la pregunta.
- **Tono pedagógico** — el dominio (enseñar IA) demanda un estilo que explique, no solo que informe.
- **Uso del historial conversacional** — se permite al modelo apoyarse en lo dicho en turnos anteriores.

In [7]:
SYSTEM_PROMPT = """Eres AsistIA, un asesor experto en Inteligencia Artificial Generativa.
Tu especialidad abarca: ingeniería de prompts, modelos de lenguaje (LLMs), técnicas RAG,
agentes de IA, frameworks como LangChain y LangGraph, y buenas prácticas del ecosistema GenAI.

INSTRUCCIONES:
1. Basa SIEMPRE tus respuestas en el contexto recuperado de la base de conocimiento.
   Si el contexto es relevante, úsalo como fuente principal.
2. Si no encuentras información suficiente en el contexto, indícalo claramente con:
   "Esta pregunta va más allá de mi base de conocimiento actual, pero puedo decirte que..."
   y proporciona una respuesta general desde tu conocimiento previo.
3. Responde SIEMPRE en español, independientemente del idioma de la pregunta.
4. Estructura tus respuestas de forma clara: usa listas cuando sea útil, destaca términos
   técnicos importantes y proporciona ejemplos concretos cuando sea posible.
5. Mantén un tono pedagógico y accesible: eres un experto que enseña, no que abruma.
6. Recuerda el contexto de la conversación: si el usuario hace referencia a algo que
   mencionó antes, úsalo para dar respuestas más personalizadas.
7. Al final de respuestas técnicas, ofrece brevemente un siguiente paso o pregunta
   de seguimiento para guiar el aprendizaje."""

print(f"System prompt configurado ({len(SYSTEM_PROMPT)} caracteres)")

System prompt configurado (1269 caracteres)


## 4. Agente LangGraph con RAG y memoria

El agente se modela como un grafo dirigido con dos nodos:

```
START → [recuperar (RAG)] → [generar (Gemini)] → END
```

- **Nodo `recuperar`** — extrae la última pregunta del usuario, consulta ChromaDB y guarda los 3 fragmentos más relevantes en el campo `contexto_rag` del estado.
- **Nodo `generar`** — inyecta `contexto_rag` y el historial completo de mensajes en el LLM para producir la respuesta.

La memoria se implementa mediante:
- El reducer `add_messages` en el campo `mensajes`, que **acumula** los turnos de conversación en lugar de reemplazarlos.
- Un `MemorySaver` como checkpointer del grafo, que persiste el estado completo asociado a cada `thread_id`.

Cada `thread_id` distinto representa una conversación independiente y aislada del resto.

In [8]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


class EstadoAsesorIA(TypedDict):
    # add_messages es el reducer que ACUMULA mensajes en lugar de reemplazarlos.
    # Esto es lo que materializa la memoria de conversación a nivel de estado.
    mensajes: Annotated[list[BaseMessage], add_messages]
    # Contexto RAG recuperado en el turno actual (se sobreescribe en cada turno).
    contexto_rag: str


# Modelo generador
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.3,        # Bajo: respuestas más precisas y consistentes
    google_api_key=API_KEY,
)

print(f"LLM inicializado: gemini-2.5-flash-lite (temperature=0.3)")

LLM inicializado: gemini-2.5-flash-lite (temperature=0.3)


### 4.1. Definición de los nodos del grafo

In [9]:
def nodo_rag(estado: EstadoAsesorIA) -> dict:
    """Recupera de ChromaDB los chunks más relevantes para la última pregunta del usuario."""
    # Localizar la última HumanMessage en el historial
    ultima_pregunta = None
    for msg in reversed(estado["mensajes"]):
        if isinstance(msg, HumanMessage):
            ultima_pregunta = msg.content
            break

    if not ultima_pregunta:
        return {"contexto_rag": ""}

    docs = retriever.invoke(ultima_pregunta)
    if not docs:
        return {"contexto_rag": "No se encontró información relevante en la base de conocimiento."}

    # Formatear los fragmentos con metadata de origen
    fragmentos = []
    for i, doc in enumerate(docs, 1):
        fuente = os.path.basename(doc.metadata.get("source", "documento"))
        pagina = doc.metadata.get("page", "?")
        fragmentos.append(f"[Fuente {i}: {fuente}, pág. {pagina}]\n{doc.page_content}")

    return {"contexto_rag": "\n\n".join(fragmentos)}


def nodo_generacion(estado: EstadoAsesorIA) -> dict:
    """Genera la respuesta inyectando el contexto RAG en el system prompt.

    El historial completo de mensajes se pasa al LLM, lo que aporta la memoria
    conversacional (gracias al reducer add_messages que ya lo ha acumulado).
    """
    contexto = estado.get("contexto_rag", "")

    system_con_contexto = SYSTEM_PROMPT
    if contexto:
        system_con_contexto += (
            f"\n\nCONTEXTO RECUPERADO DE LA BASE DE CONOCIMIENTO:\n"
            f"{'=' * 60}\n{contexto}\n{'=' * 60}\n"
            "Usa este contexto como referencia principal para tu respuesta."
        )

    mensajes_completos = [SystemMessage(content=system_con_contexto)] + estado["mensajes"]
    respuesta = llm.invoke(mensajes_completos)
    # Devolvemos solo el nuevo mensaje; el reducer lo añadirá al historial.
    return {"mensajes": [respuesta]}


print("Nodos definidos: nodo_rag → nodo_generacion")

Nodos definidos: nodo_rag → nodo_generacion


### 4.2. Construcción y compilación del grafo

In [10]:
grafo = StateGraph(EstadoAsesorIA)

grafo.add_node("recuperar", nodo_rag)
grafo.add_node("generar", nodo_generacion)

grafo.add_edge(START, "recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

# MemorySaver mantiene en memoria los checkpoints de cada thread_id.
# Cada thread_id es una conversación aislada con su propio historial.
memoria = MemorySaver()
agente = grafo.compile(checkpointer=memoria)

print("Grafo compilado:")
print("  Flujo: START → recuperar (RAG) → generar (Gemini) → END")
print("  Checkpointer: MemorySaver (memoria por thread_id)")

Grafo compilado:
  Flujo: START → recuperar (RAG) → generar (Gemini) → END
  Checkpointer: MemorySaver (memoria por thread_id)


### 4.3. Visualización del grafo

In [11]:
# Visualización del grafo. Requiere 'grandalf' para ASCII; si no está, mostramos
# las aristas de forma textual.
try:
    print(agente.get_graph().draw_ascii())
except (ImportError, Exception):
    print("Aristas del grafo:")
    for edge in agente.get_graph().edges:
        print(f"  {edge.source} → {edge.target}")

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| recuperar |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | generar |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## 5. Función de chat

`chat()` es el wrapper principal: recibe la pregunta y un `thread_id`, invoca el grafo y devuelve la respuesta. `mostrar_qa()` añade un formato legible para los ejemplos del notebook.

In [12]:
def chat(pregunta: str, thread_id: str = "sesion-1") -> str:
    """Envía una pregunta al agente dentro de un thread y devuelve la respuesta del LLM."""
    config = {"configurable": {"thread_id": thread_id}}
    entrada = {"mensajes": [HumanMessage(content=pregunta)]}
    resultado = agente.invoke(entrada, config=config)
    return resultado["mensajes"][-1].content


def mostrar_qa(pregunta: str, thread_id: str = "demo"):
    """Imprime la pregunta y la respuesta con un separador visual."""
    print("=" * 70)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 70)
    respuesta = chat(pregunta, thread_id=thread_id)
    print(f"\nASESOR IA:\n")
    print(respuesta)
    print()
    return respuesta


print("Funciones de chat listas: chat() y mostrar_qa()")

Funciones de chat listas: chat() y mostrar_qa()


## 6. Demostración con ejemplos documentados

A continuación se ejecutan **5 preguntas de ejemplo** sobre distintas áreas del dominio (prompting, modelos, RAG, agentes, recomendaciones) más una **prueba explícita de memoria conversacional**. Todas las preguntas comparten el mismo `thread_id="demo-completa"` para mantener el contexto entre turnos.

### Ejemplo 1 — Técnica zero-shot prompting

In [13]:
resp1 = mostrar_qa(
    "¿Qué es el zero-shot prompting y cuándo conviene usarlo?",
    thread_id="demo-completa"
)

PREGUNTA: ¿Qué es el zero-shot prompting y cuándo conviene usarlo?



ASESOR IA:

El **Zero-Shot Prompting** es una técnica fundamental en la ingeniería de prompts que consiste en pedirle a un modelo de lenguaje que realice una tarea **sin proporcionarle ningún ejemplo previo** de cómo hacerlo.

**¿Cuándo conviene usarlo?**

Esta técnica es especialmente útil y funciona bien en las siguientes situaciones:

*   **Tareas que el modelo ya conoce:** Si la tarea que deseas que el modelo realice es algo que probablemente haya aprendido durante su preentrenamiento a gran escala (como clasificar texto, responder preguntas generales, traducir idiomas comunes), el Zero-Shot Prompting suele ser suficiente.
*   **Simplicidad y rapidez:** Es la técnica más sencilla de implementar, ya que no requiere la preparación de ejemplos. Si necesitas una solución rápida y la tarea no es excesivamente compleja o específica, puedes empezar probando con Zero-Shot.

**Ejemplo:**

Un ejemplo clásico de Zero-Shot Prompting sería:

`Clasifica el siguiente texto como positivo, negativ

### Ejemplo 2 — Comparativa entre modelos

In [14]:
import time; time.sleep(60)
resp2 = mostrar_qa(
    "¿Cuáles son las diferencias principales entre Claude 3.5 Sonnet y GPT-4o? "
    "¿Cuál es preferible para analizar documentos largos?",
    thread_id="demo-completa"
)

PREGUNTA: ¿Cuáles son las diferencias principales entre Claude 3.5 Sonnet y GPT-4o? ¿Cuál es preferible para analizar documentos largos?



ASESOR IA:

Basándome en la información proporcionada, aquí tienes las diferencias principales entre Claude 3.5 Sonnet y GPT-4o, y cuál es preferible para analizar documentos largos:

**Diferencias Principales:**

*   **Familia y Desarrollador:**
    *   **Claude 3.5 Sonnet:** Pertenece a la familia **Anthropic**.
    *   **GPT-4o:** Pertenece a la familia **OpenAI**.

*   **Capacidades Multimodales:**
    *   **Claude 3.5 Sonnet:** El contexto proporcionado no especifica explícitamente sus capacidades multimodales, pero se enfoca en escritura, análisis de documentos, seguimiento de instrucciones complejas y código.
    *   **GPT-4o:** Es un modelo **multimodal principal** que acepta texto, imágenes y audio como entrada.

*   **Enfoque y Filosofía:**
    *   **Claude 3.5 Sonnet:** Anthropic pone un énfasis especial en la **seguridad** (mediante Constitutional AI) y en la **reducción de alucinaciones**.
    *   **GPT-4o:** Destaca en **razonamiento complejo, escritura de código, matemá

### Ejemplo 3 — Pipeline RAG y chunking

In [15]:
import time; time.sleep(60)
resp3 = mostrar_qa(
    "Explica cómo funciona el pipeline RAG paso a paso y qué es el chunking.",
    thread_id="demo-completa"
)

PREGUNTA: Explica cómo funciona el pipeline RAG paso a paso y qué es el chunking.



ASESOR IA:

¡Claro! Vamos a desglosar el pipeline RAG y el concepto de chunking.

### Pipeline RAG Paso a Paso

El **Retrieval-Augmented Generation (RAG)** es una técnica clave para mejorar las capacidades de los Modelos de Lenguaje Grandes (LLMs) al permitirles acceder y utilizar información externa y actualizada sin necesidad de reentrenarlos. El pipeline RAG consta de las siguientes fases:

1.  **Ingesta (Ingestion):**
    *   **Carga de Documentos:** Primero, se cargan los documentos que contienen el conocimiento que queremos que el LLM utilice. Estos documentos pueden ser de diversos formatos como PDF, CSV, páginas web, datos de APIs, etc.
    *   **Chunking (Segmentación):** Los documentos cargados se dividen en fragmentos más pequeños y manejables, llamados "chunks". Este proceso es crucial y lo explicaremos en detalle a continuación.
    *   **Generación de Embeddings:** A cada "chunk" se le genera una representación numérica vectorial, conocida como "embedding". Estos embeddi

### Ejemplo 4 — LangGraph vs LangChain

In [16]:
import time; time.sleep(60)
resp4 = mostrar_qa(
    "¿Para qué sirve LangGraph y en qué se diferencia de LangChain?",
    thread_id="demo-completa"
)

PREGUNTA: ¿Para qué sirve LangGraph y en qué se diferencia de LangChain?



ASESOR IA:

Basándome en el contexto proporcionado, aquí te explico para qué sirve LangGraph y cómo se diferencia de LangChain:

### ¿Para qué sirve LangGraph?

**LangGraph** es una extensión de LangChain diseñada específicamente para **construir agentes como grafos de estado dirigidos**. Su propósito principal es facilitar la creación de agentes complejos que requieren:

*   **Múltiples pasos:** Tareas que no se resuelven en una única llamada al modelo.
*   **Bifurcaciones condicionales:** Decisiones que toman el flujo de ejecución por diferentes caminos.
*   **Ciclos:** Procesos que se repiten hasta cumplir una condición (como la reflexión o la iteración).

En resumen, LangGraph es la herramienta recomendada cuando necesitas diseñar un flujo de trabajo de IA que sea más que una simple secuencia lineal de pasos. Permite modelar la lógica de un agente de manera visual y estructurada.

### ¿En qué se diferencia de LangChain?

**LangChain** es el framework más amplio y general para cons

### Ejemplo 5 — Recomendación de modelo según caso de uso

In [17]:
import time; time.sleep(60)
resp5 = mostrar_qa(
    "Para construir un chatbot de atención al cliente con alto volumen de consultas "
    "y presupuesto limitado, ¿qué modelo es más adecuado?",
    thread_id="demo-completa"
)

PREGUNTA: Para construir un chatbot de atención al cliente con alto volumen de consultas y presupuesto limitado, ¿qué modelo es más adecuado?



ASESOR IA:

Para construir un chatbot de atención al cliente con **alto volumen de consultas y presupuesto limitado**, el modelo más adecuado, según la información proporcionada, es **Claude 3 Haiku**.

Aquí te explico por qué:

*   **Bajo coste, alto volumen:** El contexto menciona explícitamente que Claude 3 Haiku es "El modelo más rápido y económico de Anthropic" y que es "ideal para tareas de producción de alto volumen". Esto se alinea perfectamente con tus requisitos de alto volumen y presupuesto limitado.
*   **Rendimiento para tareas específicas:** Se destaca que es ideal para "clasificación de tickets, resumen de emails, extracción de entidades, chatbots de primer nivel". Estas son tareas comunes y esenciales en un chatbot de atención al cliente.
*   **Ventana de contexto:** Aunque no es tan grande como otros modelos, su ventana de contexto de 200K tokens es suficiente para la mayoría de las interacciones típicas de un chatbot de atención al cliente, permitiendo analizar conve

### Ejemplo 6 — Prueba de memoria conversacional

Esta prueba está diseñada para verificar el requisito de **memoria** del enunciado: se establece un contexto personal en un thread nuevo (`demo-memoria`), se hace una pregunta intermedia y luego una **pregunta que referencia explícitamente lo dicho al principio**. Si el agente recuerda, debe mencionar el nombre y el perfil profesional aportados al inicio.

In [18]:
# Turno 1 — establecemos contexto personal
print("[Turno 1] Estableciendo contexto personal...\n")
import time
time.sleep(60)
_ = chat(
    "Me llamo Carlos. Soy desarrollador Python con experiencia en APIs REST, "
    "pero soy nuevo en IA Generativa y estoy aprendiendo a construir agentes RAG con LangGraph.",
    thread_id="demo-memoria",
)
print("Contexto inicial establecido.\n")

# Turno 2 — pregunta técnica intermedia
print("[Turno 2] Pregunta técnica intermedia...\n")
time.sleep(60)
_ = chat(
    "¿Qué embeddings me recomendarías?",
    thread_id="demo-memoria",
)
print("Respuesta intermedia recibida.\n")

# Turno 3 — pregunta que referencia el contexto del turno 1
print("[Turno 3] Pregunta que referencia el contexto inicial:\n")
time.sleep(60)
resp6 = mostrar_qa(
    "Dado el nivel y perfil que mencioné al principio, ¿por dónde me recomiendas empezar?",
    thread_id="demo-memoria",
)

print("Verificación esperada: la respuesta debe hacer referencia a 'Carlos', "
      "a su perfil de desarrollador Python y a su condición de novato en GenAI, "
      "lo que demuestra que la memoria por thread_id está funcionando.")

[Turno 1] Estableciendo contexto personal...



Contexto inicial establecido.

[Turno 2] Pregunta técnica intermedia...



Respuesta intermedia recibida.

[Turno 3] Pregunta que referencia el contexto inicial:



PREGUNTA: Dado el nivel y perfil que mencioné al principio, ¿por dónde me recomiendas empezar?



ASESOR IA:

¡Excelente pregunta, Carlos! Dado tu perfil como desarrollador Python con experiencia en APIs REST y tu interés en empezar con agentes RAG y LangGraph, te recomiendo un enfoque gradual para que construyas tu conocimiento y confianza.

Aquí te sugiero un camino de aprendizaje:

1.  **Comprender el Pipeline RAG Básico con LangChain:**
    *   **Objetivo:** Entender cómo funciona RAG en la práctica y cómo LangChain facilita su implementación.
    *   **Pasos:**
        *   **Carga de Documentos:** Empieza con un conjunto pequeño de documentos (ej. algunos archivos `.txt` o `.md`). Utiliza `DocumentLoader` de LangChain para cargarlos.
        *   **División de Texto (Chunking):** Aprende a dividir estos documentos en fragmentos manejables usando `TextSplitter`.
        *   **Generación de Embeddings:** Elige un modelo de embeddings (como los que discutimos) y úsalo para convertir los chunks en vectores.
        *   **Almacenamiento Vectorial:** Configura una base de datos vect

## 7. Celda de chat interactivo

Esta celda permite conversar libremente con el agente desde el notebook. Mantiene un mismo `thread_id` durante toda la sesión, por lo que la memoria conversacional se conserva entre preguntas. Para terminar, escribir `salir`, `exit` o pulsar Enter sin texto.

In [19]:
def chat_interactivo(thread_id: str = "interactivo"):
    """Bucle de chat por consola. Termina con 'salir'/'exit' o entrada vacía."""
    print("Chat interactivo con AsistIA")
    print(f"Thread: {thread_id}")
    print("Escribe 'salir' o pulsa Enter sin texto para terminar.\n")

    n = 0
    while True:
        try:
            pregunta = input(f"[{n+1}] Tú: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Sesión interrumpida]")
            break

        if pregunta.lower() in ("", "salir", "exit", "quit"):
            print("\n[Fin de la sesión]")
            break

        respuesta = chat(pregunta, thread_id=thread_id)
        print(f"\nAsistIA: {respuesta}\n")
        n += 1


# Para ejecutar el chat, descomenta la línea siguiente:
# chat_interactivo()
print("Función chat_interactivo() lista. Descomenta la llamada para iniciarla.")

Función chat_interactivo() lista. Descomenta la llamada para iniciarla.


## 8. Diagnóstico de la base de conocimiento

Resumen del estado de la colección vectorial y test final de recuperación con tres consultas distintas.

In [20]:
print("DIAGNÓSTICO DE LA BASE DE CONOCIMIENTO")
print("=" * 60)
print(f"Colección:               {COLLECTION_NAME}")
print(f"Total vectores:          {vectorstore._collection.count()}")
print(f"Directorio persistencia: ./{CHROMA_DIR}/")

print("\nDocumentos fuente:")
for pdf in pdfs:
    print(f"  - {os.path.basename(pdf)}")

print("\nTest de recuperación — 3 consultas:")
test_queries = [
    "few-shot prompting",
    "Gemini embeddings ChromaDB",
    "modelos abiertos LLaMA Mistral",
]
for q in test_queries:
    docs = retriever.invoke(q)
    fuentes = sorted({os.path.basename(d.metadata.get("source", "?")) for d in docs})
    print(f"  '{q}' → {len(docs)} chunks de: {', '.join(fuentes)}")

DIAGNÓSTICO DE LA BASE DE CONOCIMIENTO
Colección:               asesor_ia_generativa
Total vectores:          28
Directorio persistencia: ./chroma_db/

Documentos fuente:
  - 01_ingenieria_de_prompts.pdf
  - 02_modelos_ia_generativa.pdf
  - 03_rag_agentes_arquitecturas.pdf

Test de recuperación — 3 consultas:


  'few-shot prompting' → 3 chunks de: 01_ingenieria_de_prompts.pdf


  'Gemini embeddings ChromaDB' → 3 chunks de: 02_modelos_ia_generativa.pdf, 03_rag_agentes_arquitecturas.pdf


  'modelos abiertos LLaMA Mistral' → 3 chunks de: 02_modelos_ia_generativa.pdf
